# SignSense AI — MLP Training Notebook

**Model:** MLP landmark classifier (ASL A–Z + space/del/nothing = 29 classes)  
**Input:** 63-dim normalized MediaPipe landmark vector  
**Architecture:** Input(63) → Dense(512) → BN → ReLU → Dropout → Dense(256) → ... → Softmax(29)  
**Target accuracy:** > 95%  
**Runtime:** ~15 min on Colab T4 GPU

---
### Before you start
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your Kaggle API key (`kaggle.json`) when prompted, OR mount Google Drive with the dataset already downloaded
3. Run all cells top to bottom

In [1]:
# ── Cell 1: Mount Google Drive ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Create model output directory on Drive
import os
DRIVE_MODELS_DIR = '/content/drive/MyDrive/SignSense/models'
os.makedirs(DRIVE_MODELS_DIR, exist_ok=True)
print(f'Drive mounted. Models will be saved to: {DRIVE_MODELS_DIR}')

Mounted at /content/drive
Drive mounted. Models will be saved to: /content/drive/MyDrive/SignSense/models


In [2]:
# ── Cell 2: Install dependencies ─────────────────────────────────────────────
# Install compatible versions of mediapipe + protobuf.
# mediapipe 0.10.14 requires protobuf>=4.25 but Colab's TF 2.20 needs protobuf>=5.28.
# We pin protobuf>=5.28 and use a mediapipe version that supports it.
!pip install -q 'protobuf>=5.28.0' 'mediapipe>=0.10.18' scikit-learn tqdm albumentations

# IMPORTANT: Restart the runtime after this cell completes.
# Runtime → Restart session  (then re-run cells 1 → 2 → 3 → 4)
# This is required because protobuf is already loaded in memory.
import importlib, sys
if 'google.protobuf' in sys.modules:
    print('⚠️  protobuf already loaded in memory.')
    print('   Go to Runtime → Restart session, then re-run all cells.')
else:
    print('✅ Dependencies installed. Continue to Cell 3.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 84.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.6 MB/s eta 0:00:00
✅ Dependencies installed. Continue to Cell 3.


In [3]:
# ── Cell 3: Upload backend code to Colab ────────────────────────────────────
# Step 1: Run this PowerShell script LOCALLY to create the zip:
#   .\notebooks\create_colab_zip.ps1
#
# Step 2: Upload backend_colab.zip using the button below
#   (Files panel on the left → Upload, OR run the cell to get a file picker)

from google.colab import files
import os, sys, zipfile

BACKEND_PATH = '/content/backend'

if not os.path.exists(BACKEND_PATH):
    print('Upload backend_colab.zip when the file picker appears...')
    uploaded = files.upload()  # opens file picker
    zip_name = list(uploaded.keys())[0]
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall('/content')
    print(f'Extracted {zip_name} to /content/')
else:
    print('backend/ already exists, skipping upload.')

# Add backend to Python path
sys.path.insert(0, BACKEND_PATH)

# Verify
from configs.training_config import ASL_CLASSES, NUM_CLASSES
print(f'Import OK — {NUM_CLASSES} classes: {ASL_CLASSES[:5]}...')

Upload backend_colab.zip when the file picker appears...


Saving backend_colab.zip to backend_colab.zip
Extracted backend_colab.zip to /content/
Import OK — 29 classes: ['A', 'B', 'C', 'D', 'E']...


In [11]:
# ── Cell 4: Download & preprocess ASL dataset ─────────────────────────────────
#
# Prerequisites:
#   - Cell 3 must have run (backend/ is at /content/backend)
#   - Upload kaggle.json via Files panel BEFORE running this cell
#     Get it from: https://www.kaggle.com/settings → API → Create New Token

import os, sys, shutil, urllib.request
import numpy as np
from pathlib import Path

BACKEND_PATH   = '/content/backend'
RAW_ASL_DIR    = f'{BACKEND_PATH}/data/raw/ASL'
PROCESSED_DIR  = f'{BACKEND_PATH}/data/processed/ASL'
PROCESSED_NPY  = f'{PROCESSED_DIR}/landmarks_all.npy'
LABELS_NPY     = f'{PROCESSED_DIR}/labels_all.npy'
KAGGLE_JSON    = '/content/kaggle.json'
KAGGLE_DEST    = '/root/.kaggle/kaggle.json'
DOWNLOAD_DIR   = '/content/asl_data'
GITHUB_RAW     = 'https://raw.githubusercontent.com/prateek1756/sign-language-detection/master'

# ── Guard: backend must be uploaded first ────────────────────────────────────
if not os.path.exists(BACKEND_PATH):
    raise RuntimeError(
        'backend/ not found at /content/backend.\n'
        'Run Cell 3 first to upload backend_colab.zip.'
    )

# ── Always pull latest src files from GitHub ─────────────────────────────────
# This ensures Colab always runs the latest fixed versions regardless of
# which zip was uploaded.
SRC_FILES = [
    'backend/src/preprocess.py',
    'backend/src/model.py',
    'backend/src/train.py',
    'backend/src/evaluate.py',
    'backend/configs/training_config.py',
]
print('Pulling latest source files from GitHub...')
for rel_path in SRC_FILES:
    url  = f'{GITHUB_RAW}/{rel_path}'
    dest = f'/content/{rel_path}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(url, dest)
    print(f'  ✅ {rel_path}')

# Verify the mediapipe fix is present
with open(f'{BACKEND_PATH}/src/preprocess.py') as _f:
    assert '_USE_LEGACY_API' in _f.read(), 'preprocess.py patch missing — check GitHub'
print('Source files up to date.\n')

# ── Check if preprocessing already done ──────────────────────────────────────
if os.path.exists(PROCESSED_NPY) and os.path.exists(LABELS_NPY):
    print('✅ Preprocessed data already exists — skipping download and preprocessing.')
else:
    # ── Guard: kaggle.json must be uploaded ──────────────────────────────────
    if not os.path.exists(KAGGLE_JSON):
        raise FileNotFoundError(
            'kaggle.json not found at /content/kaggle.json.\n'
            'Steps to fix:\n'
            '  1. Go to https://www.kaggle.com/settings → API → Create New Token\n'
            '  2. This downloads kaggle.json to your computer\n'
            '  3. In Colab: Files panel (left sidebar) → Upload → select kaggle.json\n'
            '  4. Re-run this cell'
        )

    # ── Install kaggle CLI ────────────────────────────────────────────────────
    print('Installing kaggle CLI...')
    !pip install -q kaggle

    # ── Configure kaggle credentials ─────────────────────────────────────────
    os.makedirs('/root/.kaggle', exist_ok=True)
    !cp {KAGGLE_JSON} {KAGGLE_DEST}
    !chmod 600 {KAGGLE_DEST}
    print('Kaggle credentials configured.')

    # ── Download dataset ──────────────────────────────────────────────────────
    print('Downloading Kaggle ASL Alphabet dataset (~1GB)...')
    !kaggle datasets download -d grassknoted/asl-alphabet -p {DOWNLOAD_DIR} --unzip

    # ── Auto-detect actual extracted structure ────────────────────────────────
    print('\nDetecting dataset structure...')
    TRAIN_DIR = None
    best_count = 0
    for candidate in Path(DOWNLOAD_DIR).rglob('*'):
        if not candidate.is_dir():
            continue
        class_like = [d for d in candidate.iterdir()
                      if d.is_dir() and len(list(d.glob('*.jpg'))) > 0]
        if len(class_like) > best_count:
            best_count = len(class_like)
            TRAIN_DIR = str(candidate)

    if TRAIN_DIR is None or best_count == 0:
        raise FileNotFoundError(
            f'Could not find class directories with .jpg images under {DOWNLOAD_DIR}.'
        )
    print(f'✅ Detected training directory: {TRAIN_DIR}  ({best_count} classes)')

    # ── Copy raw images into project structure ────────────────────────────────
    print(f'Copying raw images to {RAW_ASL_DIR}...')
    os.makedirs(RAW_ASL_DIR, exist_ok=True)
    for class_dir in Path(TRAIN_DIR).iterdir():
        if class_dir.is_dir():
            dest = Path(RAW_ASL_DIR) / class_dir.name
            if not dest.exists():
                shutil.copytree(str(class_dir), str(dest))

    total_images = sum(
        len(list(d.glob('*.jpg')))
        for d in Path(RAW_ASL_DIR).iterdir() if d.is_dir()
    )
    print(f'✅ {total_images:,} images copied to {RAW_ASL_DIR}')
    if total_images == 0:
        raise RuntimeError(f'No .jpg images found in {RAW_ASL_DIR} after copy.')

    # ── Run preprocessing pipeline ────────────────────────────────────────────
    print('\nRunning preprocessing pipeline (~10-20 min)...')
    !python {BACKEND_PATH}/src/preprocess.py --all --augment --aug_factor 3

    if not os.path.exists(PROCESSED_NPY):
        raise RuntimeError(
            f'Preprocessing did not produce {PROCESSED_NPY}.\n'
            'Check the preprocess.py output above for errors.'
        )

    # ── Cache preprocessed data to Drive for LSTM/CNN notebooks ──────────────
    import shutil
    DRIVE_DATA = '/content/drive/MyDrive/SignSense/data'
    os.makedirs(DRIVE_DATA, exist_ok=True)
    for _f in ['landmarks_all.npy', 'labels_all.npy', 'class_map.json']:
        _src = f'{PROCESSED_DIR}/{_f}'
        _dst = f'{DRIVE_DATA}/{_f}'
        if os.path.exists(_src) and not os.path.exists(_dst):
            shutil.copy(_src, _dst)
            print(f'  ✅ Cached {_f} to Drive')
    print('Preprocessed data cached to Drive — LSTM/CNN notebooks will restore from here.')

# ── Load and verify final arrays ──────────────────────────────────────────────
X = np.load(PROCESSED_NPY)
y = np.load(LABELS_NPY)
print(f'\n✅ Data ready:')
print(f'   X shape: {X.shape}  (samples × 63 landmarks)')
print(f'   y shape: {y.shape}  (class indices 0–28)')
print(f'   Classes: {len(set(y.tolist()))} unique')
PROCESSED_NPY_PATH = PROCESSED_NPY
LABELS_NPY_PATH    = LABELS_NPY

Pulling latest source files from GitHub...
  ✅ backend/src/preprocess.py
  ✅ backend/src/model.py
  ✅ backend/src/train.py
  ✅ backend/src/evaluate.py
  ✅ backend/configs/training_config.py
Source files up to date.

✅ Preprocessed data already exists — skipping download and preprocessing.

✅ Data ready:
   X shape: (254320, 63)  (samples × 63 landmarks)
   y shape: (254320,)  (class indices 0–28)
   Classes: 29 unique


In [12]:
  # ── Cell 5: Verify GPU ────────────────────────────────────────────────────────
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print('GPU memory growth enabled.')
else:
    print('WARNING: No GPU detected. Training will be slow on CPU.')

TensorFlow version: 2.20.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU memory growth enabled.


In [13]:
# ── Cell 6: Train MLP ─────────────────────────────────────────────────────────
from pathlib import Path
from configs.training_config import MLPConfig
from src.train import train_mlp

cfg = MLPConfig()
cfg.save_dir = Path(DRIVE_MODELS_DIR)
cfg.log_dir  = Path('/content/logs/mlp')
cfg.mixed_precision = True  # Enable on T4 GPU for ~30% speedup

print('Config:')
print(f'  hidden_dims:    {cfg.hidden_dims}')
print(f'  dropout_rate:   {cfg.dropout_rate}')
print(f'  epochs:         {cfg.epochs}')
print(f'  batch_size:     {cfg.batch_size}')
print(f'  learning_rate:  {cfg.learning_rate}')
print(f'  mixed_precision:{cfg.mixed_precision}')
print()

model = train_mlp(cfg)
print('\nTraining complete!')

Config:
  hidden_dims:    [512, 256, 128]
  dropout_rate:   0.4
  epochs:         100
  batch_size:     64
  learning_rate:  0.001
  mixed_precision:True



FileNotFoundError: [Errno 2] No such file or directory: '/content/logs/mlp/config.json'

In [ ]:
# ── Cell 7: Evaluate ──────────────────────────────────────────────────────────
from src.evaluate import evaluate

# Temporarily point MODELS_DIR to Drive so evaluate() finds the model
import configs.training_config as tc
tc.MODELS_DIR = Path(DRIVE_MODELS_DIR)

results = evaluate('asl_mlp', split='test')
print(f"\nFinal test accuracy: {results['accuracy']*100:.1f}%")
print(f"Top-5 accuracy:      {results['top5']*100:.1f}%")

In [ ]:
# ── Cell 8: TensorBoard ───────────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir /content/logs/mlp

In [ ]:
# ── Cell 9: Verify saved model ────────────────────────────────────────────────
import os
saved_files = os.listdir(DRIVE_MODELS_DIR)
print('Files saved to Drive:')
for f in saved_files:
    size_mb = os.path.getsize(os.path.join(DRIVE_MODELS_DIR, f)) / 1e6
    print(f'  {f}  ({size_mb:.1f} MB)')

# Quick sanity check: load and run one prediction
import numpy as np
import tensorflow as tf
loaded = tf.keras.models.load_model(os.path.join(DRIVE_MODELS_DIR, 'asl_mlp.keras'))
dummy = np.zeros((1, 63), dtype=np.float32)
pred = loaded.predict(dummy, verbose=0)
print(f'\nSanity check — output shape: {pred.shape}  sum: {pred.sum():.4f} (should be ~1.0)')